In [1]:
# ==========================================
# CELL 1: SETUP ENVIRONMENT & DUMMY DATA
# ==========================================
# Run this cell to generate dummy dataset files if you don't have them uploaded yet.
# You can replace the text file paths with your actual dataset paths on Google Drive or Colab.

import os

os.makedirs("data/processed", exist_ok=True)
os.makedirs("results", exist_ok=True)

# Generate sample training/dev/test data to demonstrate execution
sample_text = (
    "The quick brown fox jumps over the lazy dog.\n"
    "Language models predict the probability of a sequence of words.\n"
    "Neural networks can capture long range dependencies in natural language.\n"
)

with open("data/processed/train.txt", "w", encoding="utf-8") as f:
    f.write(sample_text * 100)

with open("data/processed/valid.txt", "w", encoding="utf-8") as f:
    f.write(sample_text * 20)

with open("data/processed/test.txt", "w", encoding="utf-8") as f:
    f.write(sample_text * 20)

print("Setup completed and dummy data generated!")

Setup completed and dummy data generated!


In [ ]:
# ==========================================
# CELL 2: SHARED UTILITIES & TOKENIZER
# ==========================================
import re
import math
from collections import Counter, defaultdict

# Tokenization Pattern
TOKEN_PATTERN = re.compile(r"\w+(?:'\w+)?|[^\w\s]")

def tokenize(text):
    """Convert a string to a lowercase sequence of word/punctuation tokens."""
    normalized = text.lower().replace("’", "'")
    return TOKEN_PATTERN.findall(normalized)

def compute_perplexity(avg_cross_entropy_bits):
    """Converts average cross-entropy (in bits) to perplexity."""
    return 2 ** avg_cross_entropy_bits

In [ ]:
# ==========================================
# CELL 3: SECTION 1 - N-GRAM LANGUAGE MODEL
# ==========================================
START = "<s>"
END = "</s>"

def with_boundaries(sentence, n=2):
    """Tokenize a sentence and add start/end symbols for an order-n model."""
    tokens = tokenize(sentence)
    return [START] * (n - 1) + tokens + [END]

def build_ngram_counts(sentences, n):
    next_counts = defaultdict(Counter)
    history_counts = Counter()
    vocab = set()
    
    for sentence in sentences:
        tokens = with_boundaries(sentence, n=n)
        vocab.update(tokens)
        
        for position in range(n - 1, len(tokens)):
            history = tuple(tokens[position - n + 1 : position])
            word = tokens[position]
            next_counts[history][word] += 1
            history_counts[history] += 1
            
    return next_counts, history_counts, vocab

class CountLanguageModel:
    def __init__(self, max_order):
        self.max_order = max_order
        self.next_counts = {}
        self.history_counts = {}
        self.vocabulary = set()

    def fit(self, sentences):
        for order in range(1, self.max_order + 1):
            next_counts, history_counts, vocab = build_ngram_counts(sentences, order)
            self.next_counts[order] = next_counts
            self.history_counts[order] = history_counts
            self.vocabulary.update(vocab)
        return self

    def history_for(self, history_tokens, order):
        if order == 1:
            return ()
        padded = [START] * max(0, order - 1 - len(history_tokens)) + list(history_tokens)
        return tuple(padded[-(order - 1):])

    def probability(self, history_tokens, word, order, add_k=0.0):
        history = self.history_for(history_tokens, order)
        
        # Unsmoothed N-gram logic
        if add_k == 0.0:
            history_count = self.history_counts[order][history]
            if history_count == 0:
                return 0.0
            return self.next_counts[order][history][word] / history_count

        # Add-k Smoothed N-gram logic
        numerator = self.next_counts[order][history][word] + add_k
        denominator = self.history_counts[order][history] + (add_k * len(self.vocabulary))
        return numerator / denominator if denominator > 0 else 0.0

    def interpolated_probability(self, history_tokens, word, weights, add_k=0.0):
        prob = 0.0
        for order, w in weights.items():
            prob += w * self.probability(history_tokens, word, order, add_k=add_k)
        return prob

def evaluate_ngram_model(model, sentences, weights, add_k):
    """Calculates cross-entropy loss and perplexity on a given dataset."""
    log_prob_sum = 0.0
    token_count = 0
    
    for sentence in sentences:
        tokens = with_boundaries(sentence, n=model.max_order)
        for position in range(model.max_order - 1, len(tokens)):
            history_tokens = tokens[position - model.max_order + 1 : position]
            word = tokens[position]
            
            prob = model.interpolated_probability(
                history_tokens, word, weights=weights, add_k=add_k
            )
            
            prob = max(prob, 1e-12)
            log_prob_sum += math.log2(prob)
            token_count += 1

    if token_count == 0:
        return float('inf'), float('inf')

    avg_cross_entropy = -log_prob_sum / token_count
    perplexity = 2 ** avg_cross_entropy
    return avg_cross_entropy, perplexity

In [ ]:
# ==========================================
# CELL 4: TRAIN & EVALUATE N-GRAM MODEL
# ==========================================
# Load Datasets
with open("data/processed/train.txt", "r", encoding="utf-8") as f:
    train_sentences = f.readlines()
with open("data/processed/valid.txt", "r", encoding="utf-8") as f:
    dev_sentences = f.readlines()
with open("data/processed/test.txt", "r", encoding="utf-8") as f:
    test_sentences = f.readlines()

# Fit N-Gram Model
max_order = 3
ngram_model = CountLanguageModel(max_order=max_order)
ngram_model.fit(train_sentences)

weights = {order: 1.0 / max_order for order in range(1, max_order + 1)}

# Hyperparameter Tuning
candidate_k_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
best_k = None
best_dev_pp = float('inf')

print("--- Tuning Hyperparameters on Dev Set ---")
for k in candidate_k_values:
    _, dev_pp = evaluate_ngram_model(ngram_model, dev_sentences, weights=weights, add_k=k)
    print(f"add_k = {k:<5} | Dev Perplexity: {dev_pp:.4f}")
    
    if dev_pp < best_dev_pp:
        best_dev_pp = dev_pp
        best_k = k

print(f"\nBest add_k found: {best_k} (Dev Perplexity: {best_dev_pp:.4f})")

# Final Evaluation
test_ce, test_pp = evaluate_ngram_model(ngram_model, test_sentences, weights=weights, add_k=best_k)

print("\n--- Final N-Gram Test Set Results ---")
print(f"Test Cross-Entropy: {test_ce:.4f} bits/token")
print(f"Test Perplexity:    {test_pp:.4f}")

In [ ]:
# ==========================================
# CELL 5: SECTION 2 - RNN MODEL ARCHITECTURE & DATA PREPARATION
# ==========================================
import copy
import random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

class VanillaRNNLM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embeds = self.embedding(x)
        out, hidden = self.rnn(embeds, hidden)
        logits = self.fc(out)
        return logits, hidden

def read_tokens(path):
    text = Path(path).read_text(encoding="utf-8")
    return tokenize(text)

def build_vocabulary(tokens):
    vocabulary = [PAD_TOKEN, UNK_TOKEN]
    vocabulary.extend(sorted(set(tokens) - {PAD_TOKEN, UNK_TOKEN}))
    token_to_id = {token: index for index, token in enumerate(vocabulary)}
    return token_to_id, vocabulary

def make_data_loader(tokens, token_to_id, sequence_length, batch_size, shuffle):
    unknown_id = token_to_id[UNK_TOKEN]
    ids = [token_to_id.get(token, unknown_id) for token in tokens]
    inputs, targets = [], []

    for start in range(0, len(ids) - sequence_length, sequence_length):
        window = ids[start : start + sequence_length + 1]
        if len(window) == sequence_length + 1:
            inputs.append(window[:-1])
            targets.append(window[1:])

    if not inputs:
        raise ValueError("The split is too short for the selected sequence length.")

    dataset = TensorDataset(
        torch.tensor(inputs, dtype=torch.long),
        torch.tensor(targets, dtype=torch.long),
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def evaluate_neural_model(model, data_loader, criterion, device):
    """Evaluates a PyTorch Neural LM (RNN, LSTM, Transformer)."""
    model.eval()
    total_loss_nats = 0.0
    total_tokens = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            if isinstance(outputs, tuple):
                outputs = outputs[0]
            
            vocab_size = outputs.size(-1)
            outputs = outputs.view(-1, vocab_size)
            targets = targets.view(-1)
            
            loss = criterion(outputs, targets) 
            non_pad_tokens = (targets != criterion.ignore_index).sum().item()
            
            total_loss_nats += loss.item()
            total_tokens += non_pad_tokens

    if total_tokens == 0:
        return float('inf'), float('inf')

    avg_cross_entropy_bits = (total_loss_nats / total_tokens) / math.log(2)
    ppl = compute_perplexity(avg_cross_entropy_bits)
    
    return avg_cross_entropy_bits, ppl

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
# ==========================================
# CELL 6: RNN TRAINING FUNCTION
# ==========================================
def train_rnn_model(
    model, train_loader, valid_loader, criterion, optimizer, device, epochs, patience
):
    model.to(device)
    best_valid_ppl = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_tokens = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            logits, _ = model(inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * targets.numel()
            total_tokens += targets.numel()

        train_loss = total_loss / total_tokens
        valid_bits, valid_ppl = evaluate_neural_model(
            model, valid_loader, criterion, device
        )
        print(
            f"Epoch {epoch:02d} | train loss: {train_loss:.4f} nats/token | "
            f"valid CE: {valid_bits:.4f} bits/token | valid PPL: {valid_ppl:.4f}"
        )

        if valid_ppl < best_valid_ppl:
            best_valid_ppl = valid_ppl
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping after {epoch} epochs.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_valid_ppl

In [ ]:
# ==========================================
# CELL 7: RUN RNN TRAINING & EVALUATION
# ==========================================
# Model Parameters & Configs
class Config:
    train_path = "data/processed/train.txt"
    valid_path = "data/processed/valid.txt"
    test_path = "data/processed/test.txt"
    checkpoint = "results/rnn.pt"
    epochs = 5
    batch_size = 64
    sequence_length = 32
    embed_dim = 128
    hidden_dim = 256
    learning_rate = 1e-3
    patience = 2
    seed = 42

args = Config()
set_seed(args.seed)

# Data Processing
train_tokens = read_tokens(args.train_path)
valid_tokens = read_tokens(args.valid_path)
test_tokens = read_tokens(args.test_path)

token_to_id, vocabulary = build_vocabulary(train_tokens)

train_loader = make_data_loader(train_tokens, token_to_id, args.sequence_length, args.batch_size, shuffle=True)
valid_loader = make_data_loader(valid_tokens, token_to_id, args.sequence_length, args.batch_size, shuffle=False)
test_loader = make_data_loader(test_tokens, token_to_id, args.sequence_length, args.batch_size, shuffle=False)

# Model Instantiation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rnn_model = VanillaRNNLM(len(vocabulary), args.embed_dim, args.hidden_dim)
criterion = nn.CrossEntropyLoss(ignore_index=token_to_id[PAD_TOKEN], reduction="sum")
optimizer = torch.optim.Adam(rnn_model.parameters(), lr=args.learning_rate)

print(f"Vocabulary: {len(vocabulary):,} tokens | device: {device}")

# Train
best_valid_ppl = train_rnn_model(
    rnn_model, train_loader, valid_loader, criterion, optimizer, device, args.epochs, args.patience
)

# Test
test_bits, test_ppl = evaluate_neural_model(rnn_model, test_loader, criterion, device)
print(f"\n--- Final RNN Test Set Results ---")
print(f"Best valid PPL: {best_valid_ppl:.4f} | test CE: {test_bits:.4f} bits/token | test PPL: {test_ppl:.4f}")

# Save Checkpoint
checkpoint_path = Path(args.checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": rnn_model.state_dict(),
        "token_to_id": token_to_id,
        "vocabulary": vocabulary,
        "model_config": {
            "embed_dim": args.embed_dim,
            "hidden_dim": args.hidden_dim,
        },
    },
    checkpoint_path,
)
print(f"Saved checkpoint to {checkpoint_path}")

In [ ]:
# ==========================================
# CELL 8: SECTION 3 - LSTM ARCHITECTURE & TRAINING SETUP
# ==========================================
class LSTMLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(
            embed_dim, 
            hidden_dim, 
            num_layers=num_layers, 
            batch_first=True, 
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        embeds = self.embedding(x)
        out, hidden = self.lstm(embeds, hidden)
        logits = self.fc(out)
        return logits, hidden

def train_lstm_model(
    model, train_loader, valid_loader, criterion, optimizer, device, epochs, patience
):
    model.to(device)
    best_valid_ppl = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_tokens = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            logits, _ = model(inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            total_loss += loss.item() * targets.numel()
            total_tokens += targets.numel()

        train_loss = total_loss / total_tokens
        valid_bits, valid_ppl = evaluate_neural_model(
            model, valid_loader, criterion, device
        )
        print(
            f"Epoch {epoch:02d} | train loss: {train_loss:.4f} nats/token | "
            f"valid CE: {valid_bits:.4f} bits/token | valid PPL: {valid_ppl:.4f}"
        )

        if valid_ppl < best_valid_ppl:
            best_valid_ppl = valid_ppl
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping after {epoch} epochs.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_valid_ppl

In [ ]:
# ==========================================
# CELL 9: RUN LSTM TRAINING & EVALUATION
# ==========================================
class LSTMConfig:
    train_path = "data/processed/train.txt"
    valid_path = "data/processed/valid.txt"
    test_path = "data/processed/test.txt"
    checkpoint = "results/lstm.pt"
    epochs = 5
    batch_size = 64
    sequence_length = 32
    embed_dim = 128
    hidden_dim = 256
    learning_rate = 1e-3
    patience = 2
    seed = 42

lstm_args = LSTMConfig()
set_seed(lstm_args.seed)

# Data Processing
train_tokens = read_tokens(lstm_args.train_path)
valid_tokens = read_tokens(lstm_args.valid_path)
test_tokens = read_tokens(lstm_args.test_path)

token_to_id, vocabulary = build_vocabulary(train_tokens)

train_loader = make_data_loader(train_tokens, token_to_id, lstm_args.sequence_length, lstm_args.batch_size, shuffle=True)
valid_loader = make_data_loader(valid_tokens, token_to_id, lstm_args.sequence_length, lstm_args.batch_size, shuffle=False)
test_loader = make_data_loader(test_tokens, token_to_id, lstm_args.sequence_length, lstm_args.batch_size, shuffle=False)

# Model Instantiation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lstm_model = LSTMLanguageModel(len(vocabulary), lstm_args.embed_dim, lstm_args.hidden_dim)
criterion = nn.CrossEntropyLoss(ignore_index=token_to_id[PAD_TOKEN], reduction="sum")
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=lstm_args.learning_rate)

print(f"Vocabulary: {len(vocabulary):,} tokens | device: {device}")

# Train
best_valid_ppl = train_lstm_model(
    lstm_model, train_loader, valid_loader, criterion, optimizer, device, lstm_args.epochs, lstm_args.patience
)

# Test
test_bits, test_ppl = evaluate_neural_model(lstm_model, test_loader, criterion, device)
print(f"\n--- Final LSTM Test Set Results ---")
print(f"Best valid PPL: {best_valid_ppl:.4f} | test CE: {test_bits:.4f} bits/token | test PPL: {test_ppl:.4f}")

# Save Checkpoint
checkpoint_path = Path(lstm_args.checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": lstm_model.state_dict(),
        "token_to_id": token_to_id,
        "vocabulary": vocabulary,
        "model_config": {
            "embed_dim": lstm_args.embed_dim,
            "hidden_dim": lstm_args.hidden_dim,
        },
    },
    checkpoint_path,
)
print(f"Saved checkpoint to {checkpoint_path}")

In [ ]:
# ==========================================
# CELL 10: SECTION 4 - TRANSFORMER ARCHITECTURE & TRAINING SETUP
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        return x + self.pe[:, :x.size(1)]


class TransformerLanguageModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, nhead, hidden_dim, num_layers, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoder = PositionalEncoding(embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=nhead, 
            dim_feedforward=hidden_dim, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, vocab_size)

    def _generate_square_subsequent_mask(self, sz, device):
        """Causal mask to prevent attending to future tokens."""
        mask = torch.triu(torch.full((sz, sz), float('-inf'), device=device), diagonal=1)
        return mask

    def forward(self, x):
        # Scale embedding by sqrt(d_model) following Vaswani et al.
        x = self.embedding(x) * math.sqrt(self.embed_dim)
        x = self.pos_encoder(x)
        
        mask = self._generate_square_subsequent_mask(x.size(1), x.device)
        out = self.transformer(x, mask=mask, is_causal=True)
        logits = self.fc(out)
        return logits, None


def train_transformer_model(
    model, train_loader, valid_loader, criterion, optimizer, device, epochs, patience
):
    model.to(device)
    best_valid_ppl = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_tokens = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            logits, _ = model(inputs)
            loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item() * targets.numel()
            total_tokens += targets.numel()

        train_loss = total_loss / total_tokens
        valid_bits, valid_ppl = evaluate_neural_model(
            model, valid_loader, criterion, device
        )
        print(
            f"Epoch {epoch:02d} | train loss: {train_loss:.4f} nats/token | "
            f"valid CE: {valid_bits:.4f} bits/token | valid PPL: {valid_ppl:.4f}"
        )

        if valid_ppl < best_valid_ppl:
            best_valid_ppl = valid_ppl
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping after {epoch} epochs.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return best_valid_ppl

In [ ]:
# ==========================================
# CELL 11: RUN TRANSFORMER TRAINING & EVALUATION
# ==========================================
class TransformerConfig:
    train_path = "data/processed/train.txt"
    valid_path = "data/processed/valid.txt"
    test_path = "data/processed/test.txt"
    checkpoint = "results/transformer.pt"
    epochs = 5
    batch_size = 64
    sequence_length = 32
    embed_dim = 128
    nhead = 4            # Number of multi-head attention heads
    hidden_dim = 256      # Feedforward network dimension
    num_layers = 2       # Number of Transformer blocks
    learning_rate = 1e-3
    patience = 2
    seed = 42

tx_args = TransformerConfig()
set_seed(tx_args.seed)

# Data Processing
train_tokens = read_tokens(tx_args.train_path)
valid_tokens = read_tokens(tx_args.valid_path)
test_tokens = read_tokens(tx_args.test_path)

token_to_id, vocabulary = build_vocabulary(train_tokens)

train_loader = make_data_loader(train_tokens, token_to_id, tx_args.sequence_length, tx_args.batch_size, shuffle=True)
valid_loader = make_data_loader(valid_tokens, token_to_id, tx_args.sequence_length, tx_args.batch_size, shuffle=False)
test_loader = make_data_loader(test_tokens, token_to_id, tx_args.sequence_length, tx_args.batch_size, shuffle=False)

# Model Instantiation
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tx_model = TransformerLanguageModel(
    vocab_size=len(vocabulary),
    embed_dim=tx_args.embed_dim,
    nhead=tx_args.nhead,
    hidden_dim=tx_args.hidden_dim,
    num_layers=tx_args.num_layers
)
criterion = nn.CrossEntropyLoss(ignore_index=token_to_id[PAD_TOKEN], reduction="sum")
optimizer = torch.optim.Adam(tx_model.parameters(), lr=tx_args.learning_rate)

print(f"Vocabulary: {len(vocabulary):,} tokens | device: {device}")

# Train
best_valid_ppl = train_transformer_model(
    tx_model, train_loader, valid_loader, criterion, optimizer, device, tx_args.epochs, tx_args.patience
)

# Test
test_bits, test_ppl = evaluate_neural_model(tx_model, test_loader, criterion, device)
print(f"\n--- Final Transformer Test Set Results ---")
print(f"Best valid PPL: {best_valid_ppl:.4f} | test CE: {test_bits:.4f} bits/token | test PPL: {test_ppl:.4f}")

# Save Checkpoint
checkpoint_path = Path(tx_args.checkpoint)
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "model_state_dict": tx_model.state_dict(),
        "token_to_id": token_to_id,
        "vocabulary": vocabulary,
        "model_config": {
            "embed_dim": tx_args.embed_dim,
            "hidden_dim": tx_args.hidden_dim,
            "nhead": tx_args.nhead,
            "num_layers": tx_args.num_layers,
        },
    },
    checkpoint_path,
)
print(f"Saved checkpoint to {checkpoint_path}")